In [1]:
#Import the required Libraries
import os
import numpy as np
from openai import OpenAI
import faiss
from dotenv import load_dotenv
import PyPDF2
import json
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
#Function to read data from the input PDF file and save it to text

def load_pdf_to_text(path):
    reader = PyPDF2.PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

pdf_text = load_pdf_to_text("Policy & Claims Copilot – Knowledge Base.pdf")
print(pdf_text[:500])


Policy & Claims Copilot – Knowledge Base
Document
1. Overview
The Policy & Claims Copilot  is designed to assist customers, insurance agents, and claims teams by
providing accurate, consistent, and real-time information about insurance policies and claims processes.
This system leverages Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG) to answer
queries and perform preliminary claim validations.
2. Policy Coverage
2.1 What is Covered
Coverage varies by policy type but typica


In [3]:
#Create OPEN-AI Client
load_dotenv(dotenv_path=r'C:\Users\User\Agentic AI\.env')
openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key = openai_api_key,
                       project="proj_AiRCDdpGwAmt6xebs4kjwD8f" )

In [4]:
#Define fucntion to call a open-api embeddings model to perform word embeddings
def create_embeddings(text):
    response = openai_client.embeddings.create(
        model = "text-embedding-3-small",
        input = text
    )
    return np.array(response.data[0].embedding, dtype="float32")


In [5]:
#Function to chunk the text usint NLTK punkt, here we are creating chucks of 500 characters
def chunk_by_sent(text, max_chars=500):
    sentences = nltk.sent_tokenize(text)
    chunks = []
    current = ""

    for sentence in sentences:
        if len(current) + len(sentence) < max_chars:
            current += " " + sentence
        else:
            chunks.append(current.strip())
            current = sentence
    if current:
        chunks.append(current.strip())

    return chunks


In [6]:
#Chunk text nefore proceeding to Embedding in order to give better context to RAG
chunks = chunk_by_sent(pdf_text)
print(chunks[:3])

['Policy & Claims Copilot – Knowledge Base\nDocument\n1. Overview\nThe Policy & Claims Copilot  is designed to assist customers, insurance agents, and claims teams by\nproviding accurate, consistent, and real-time information about insurance policies and claims processes. This system leverages Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG) to answer\nqueries and perform preliminary claim validations. 2.', 'Policy Coverage\n2.1 What is Covered\nCoverage varies by policy type but typically includes:\nHospitalization expenses (room rent, ICU, nursing)\nPre- and post-hospitalization costs (e.g., consultations, diagnostics)\nDaycare procedures not requiring 24-hour admission\nEmergency ambulance services\nCertain critical illnesses (as specified in policy documents)\nMaternity benefits (if included)\n2.2 What is Not Covered\nCommon exclusions include:\nPre-existing diseases (during waiting period)\nCosmetic or aesthetic treatments\nSelf-inflicted injuries or suicide a

In [7]:
#Create word embedding and save them in faiss index
# dimension of embedding
d = 1536
# create index
index = faiss.IndexFlatL2(d)
# metadata store (FAISS cannot store text)
metadata = []

#create embedding and save in the Faiss Index
#Create word embeddings and save index & text in faiss index & metadata
for i, t in enumerate(chunks):
    vec = create_embeddings(t)
    index.add(vec.reshape(1, -1))# FAISS expects shape (1, d)
    metadata.append({"id": i, "text": t})

In [8]:
#Write data to faiss index
faiss.write_index(index, "policy_index.faiss")
#To save the text into the metadata.json file
with open("metadata.json", "w") as f:
    json.dump(metadata, f)

In [9]:
#Load the index & metadata into memory to be available for LLM/RAG
index = faiss.read_index("policy_index.faiss")
with open("metadata.json", "r") as f:
    metadata = json.load(f)

In [10]:
#Function to embed the user query, retrieve the information from the faiss & share it as context to the LLM and run the LLM 
def ask_policy_copilot(query):
    #query = "How do i raise a claim?"
    query_vector = create_embeddings(query).reshape(1, -1)

    k = 2  # number of results
    distances, indices = index.search(query_vector, k)
    retrieved_data = []

    for i in indices[0]:
        retrieved_data.append(metadata[i]["text"])

    #Saving the FAISS response as Context to the LLM
    retrieved_Response = "\n\n".join(retrieved_data)

    prompt = f"""
    You're are COPAC:Policy Claim Co-pilot. Use ONLY the context below to answer the question.

    Context:
    {retrieved_Response}

    Question:
    {query}

    Answer:
    """
    response = openai_client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}])

    return response.choices[0].message.content

In [11]:
while True:
    query = input("Am COPAC: Policy & Claims Co-pilot. Please enter your query")
    copilt_response =ask_policy_copilot(query)
    print(f"\n Your Query: {query}\n Copilot_response:\n {copilt_response}")
    if query.lower() == "quit":
        break


 Your Query: How to raise a claim?
 Copilot_response:
 Steps to raise a claim

1. Notify the insurer or TPA (Third Party Administrator) as soon as possible.  
2. Submit the claim form (online or offline).  
3. Provide the required documents.  
4. Undergo the claim assessment.  
5. Receive the approval or rejection decision.

Types of claim settlements
- Cashless claims: Direct settlement with a network hospital.  
- Reimbursement claims: You pay upfront and claim the costs later.

Note: Make sure your policy is active and any waiting period (if applicable) has been completed.

 Your Query: How to raise a cash-less claim?
 Copilot_response:
 Steps to raise a cashless claim (using only the provided context)

1. Use a network hospital — cashless settlement is available only at network hospitals.  
2. Notify the insurer or TPA (Third Party Administrator) as soon as possible.  
3. Submit the claim form (online or offline) to the insurer/TPA or hospital as instructed.  
4. Provide the requi